In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

batch_size = 2
seq_len = 4
d_model = 8
num_heads = 2

In [3]:
X = torch.randn(batch_size, seq_len, d_model)

print(X.shape)
print(X[1])

torch.Size([2, 4, 8])
tensor([[-2.5095e+00,  4.8800e-01,  7.8459e-01,  2.8647e-02,  6.4076e-01,
          5.8325e-01,  1.0669e+00, -4.5015e-01],
        [-1.8527e-01,  7.5276e-01,  4.0476e-01,  1.7847e-01,  2.6491e-01,
          1.2732e+00, -1.3109e-03, -3.0360e-01],
        [-1.4570e+00, -1.0234e-01, -5.9915e-01,  4.7706e-01,  7.2618e-01,
          9.1152e-02, -3.8907e-01,  5.2792e-01],
        [-1.2685e-02,  2.4084e-01,  1.3254e-01,  7.6424e-01,  1.0950e+00,
          3.3989e-01,  7.1997e-01,  4.1141e-01]])


In [5]:
X[1 , -1] = 0

X[1 ,-1]

tensor([0., 0., 0., 0., 0., 0., 0., 0.])

In [6]:
valid_mask = torch.tensor([
    [True, True, True, True],
    [True, True, True, False]
])

print(valid_mask.shape)
print(valid_mask.dtype)

torch.Size([2, 4])
torch.bool


In [10]:
assert d_model % num_heads == 0
print("Test 1 —  PASSED")


head_dim = d_model // num_heads

print("Head Dimension:", head_dim)
print("Head Dimension Type:", type(head_dim))

Test 1 —  PASSED
Head Dimension: 4
Head Dimension Type: <class 'int'>


In [11]:

W_q = nn.Linear(d_model, d_model, bias=False)

W_k = nn.Linear(d_model, d_model, bias=False)

W_v = nn.Linear(d_model, d_model, bias=False)


Q = W_q(X)
K = W_k(X)
V = W_v(X)

print("X:", X.shape)
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

X: torch.Size([2, 4, 8])
Q: torch.Size([2, 4, 8])
K: torch.Size([2, 4, 8])
V: torch.Size([2, 4, 8])


In [12]:
Q_reshaped = Q.reshape(batch_size, seq_len, num_heads, head_dim)

print("Original Q:", Q.shape)
print("Reshaped Q:", Q_reshaped.shape)

Original Q: torch.Size([2, 4, 8])
Reshaped Q: torch.Size([2, 4, 2, 4])


In [16]:
Q_heads = Q_reshaped.transpose(1, 2)

print("Q Heads Shape:", Q_heads.shape)

Q Heads Shape: torch.Size([2, 2, 4, 4])


In [17]:
K_reshaped = K.reshape(batch_size, seq_len, num_heads, head_dim)
K_heads = K_reshaped.transpose(1, 2)

V_reshaped = V.reshape(batch_size, seq_len, num_heads, head_dim)
V_heads = V_reshaped.transpose(1, 2)

In [18]:
print("Q Heads:", Q_heads.shape)
print("K Heads:", K_heads.shape)
print("V Heads:", V_heads.shape)

Q Heads: torch.Size([2, 2, 4, 4])
K Heads: torch.Size([2, 2, 4, 4])
V Heads: torch.Size([2, 2, 4, 4])


In [21]:
K_T = K_heads.transpose(-2 , -1)

scores = torch.matmul(Q_heads , K_T)

print("K Transpose:", K_T.shape)
print("Attention Scores:", scores.shape)

K Transpose: torch.Size([2, 2, 4, 4])
Attention Scores: torch.Size([2, 2, 4, 4])


In [24]:
scaled_scores =  scores / math.sqrt(head_dim)

print("Original Scores:", scores.shape)
print("Scaled Scores:", scaled_scores.shape)
print("First Head Scores:", scaled_scores[0, 0])

Original Scores: torch.Size([2, 2, 4, 4])
Scaled Scores: torch.Size([2, 2, 4, 4])
First Head Scores: tensor([[ 0.4069, -0.5455,  0.3159,  0.5239],
        [ 0.1788, -0.1271, -0.0241,  0.2416],
        [-0.2918,  0.1555, -0.0128, -0.2225],
        [-0.2245,  0.0027,  0.2381, -0.0121]], grad_fn=<SelectBackward0>)


In [32]:
key_mask = valid_mask.unsqueeze(1).unsqueeze(1)

print("Original Mask:", valid_mask.shape)
print("Expanded Mask:", key_mask.shape)

Original Mask: torch.Size([2, 4])
Expanded Mask: torch.Size([2, 1, 1, 4])


In [33]:
masked_scores = scaled_scores.masked_fill(
    ~key_mask,
    float("-inf")
)

print("Masked Scores Shape:", masked_scores.shape)

print("Head 1 PAD Scores:")
print(masked_scores[1, 0, :, 3])

print("Head 2 PAD Scores:")
print(masked_scores[1, 1, :, 3])

Masked Scores Shape: torch.Size([2, 2, 4, 4])
Head 1 PAD Scores:
tensor([-inf, -inf, -inf, -inf], grad_fn=<SelectBackward0>)
Head 2 PAD Scores:
tensor([-inf, -inf, -inf, -inf], grad_fn=<SelectBackward0>)


In [38]:
attention_weights = F.softmax(
    masked_scores , 
    dim=-1
)

print("Attention Weights Shape:", attention_weights.shape)

print("Head 1 PAD Weights:")
print(attention_weights[1, 0, :, 3])

print("Head 2 PAD Weights:")
print(attention_weights[1, 1, :, 3])


print("Attention Weights Sum:")
print(attention_weights.sum(dim=-1))

Attention Weights Shape: torch.Size([2, 2, 4, 4])
Head 1 PAD Weights:
tensor([0., 0., 0., 0.], grad_fn=<SelectBackward0>)
Head 2 PAD Weights:
tensor([0., 0., 0., 0.], grad_fn=<SelectBackward0>)
Attention Weights Sum:
tensor([[[1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)


In [39]:
head_outputs = torch.matmul(attention_weights , V_heads )

print("Attention Weights:", attention_weights.shape)
print("V Heads:", V_heads.shape)
print("Head Outputs:", head_outputs.shape)

Attention Weights: torch.Size([2, 2, 4, 4])
V Heads: torch.Size([2, 2, 4, 4])
Head Outputs: torch.Size([2, 2, 4, 4])


In [41]:
transposed_output = head_outputs.transpose(1,2)


contiguous_output = transposed_output.contiguous()

concatenated_output = contiguous_output.reshape(batch_size , seq_len , d_model)

In [42]:
print("Transposed:", transposed_output.shape)
print("Contiguous:", contiguous_output.shape)
print("Concatenated:", concatenated_output.shape)

Transposed: torch.Size([2, 4, 2, 4])
Contiguous: torch.Size([2, 4, 2, 4])
Concatenated: torch.Size([2, 4, 8])


In [43]:
W_o = nn.Linear(d_model, d_model, bias=False)

multihead_output = W_o(concatenated_output)

print("Concatenated:", concatenated_output.shape)
print("Multi-Head Output:", multihead_output.shape)

Concatenated: torch.Size([2, 4, 8])
Multi-Head Output: torch.Size([2, 4, 8])


In [49]:
def positional_encoding(seq_len, d_model):

    positions = torch.arange(seq_len).unsqueeze(1)

    div_term = torch.exp(
        torch.arange(0, d_model, 2).float()
        * (-math.log(10000.0) / d_model)
    )
    angles = positions * div_term

    PE = torch.zeros(seq_len , d_model)
    PE[:, 0::2] = torch.sin(angles)
    PE[:, 1::2] = torch.cos(angles)

    return PE

PE = positional_encoding(4, 8)

print("Positional Encoding Shape:", PE.shape)
print(PE)

Positional Encoding Shape: torch.Size([4, 8])
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00]])


In [50]:
print("Position 0:", PE[0])
print("Position 1:", PE[1])
print("Position 2:", PE[2])
print("Position 3:", PE[3])

Position 0: tensor([0., 1., 0., 1., 0., 1., 0., 1.])
Position 1: tensor([0.8415, 0.5403, 0.0998, 0.9950, 0.0100, 0.9999, 0.0010, 1.0000])
Position 2: tensor([ 0.9093, -0.4161,  0.1987,  0.9801,  0.0200,  0.9998,  0.0020,  1.0000])
Position 3: tensor([ 0.1411, -0.9900,  0.2955,  0.9553,  0.0300,  0.9996,  0.0030,  1.0000])


In [51]:
X_pos = X + PE.unsqueeze(0)

print("Original Input Shape:", X.shape)
print("Positional Encoding Shape:", PE.shape)
print("New Input Shape:", X_pos.shape)

Original Input Shape: torch.Size([2, 4, 8])
Positional Encoding Shape: torch.Size([4, 8])
New Input Shape: torch.Size([2, 4, 8])


In [52]:
print("Original First Token:")
print(X[0, 0])

print("\nWith Positional Encoding:")
print(X_pos[0, 0])

Original First Token:
tensor([ 1.4451,  0.8564,  2.2181,  0.5232,  0.3466, -0.1973, -1.0546,  1.2780])

With Positional Encoding:
tensor([ 1.4451,  1.8564,  2.2181,  1.5232,  0.3466,  0.8027, -1.0546,  2.2780])


In [53]:
X_pos = X + PE.unsqueeze(0)

print("X Shape:", X.shape)
print("PE Shape:", PE.shape)
print("X_pos Shape:", X_pos.shape)

print("\nOriginal PAD:")
print(X[1, -1])

print("\nPAD With Positional Encoding:")
print(X_pos[1, -1])

X Shape: torch.Size([2, 4, 8])
PE Shape: torch.Size([4, 8])
X_pos Shape: torch.Size([2, 4, 8])

Original PAD:
tensor([0., 0., 0., 0., 0., 0., 0., 0.])

PAD With Positional Encoding:
tensor([ 0.1411, -0.9900,  0.2955,  0.9553,  0.0300,  0.9996,  0.0030,  1.0000])


In [54]:
Q_A = W_q(X)
K_A = W_k(X)
V_A = W_v(X)


Q_B = W_q(X_pos)
K_B = W_k(X_pos)
V_B = W_v(X_pos)


print("Q_A Shape:", Q_A.shape)
print("Q_B Shape:", Q_B.shape)

print("Q değerleri aynı mı?", torch.allclose(Q_A, Q_B))

Q_A Shape: torch.Size([2, 4, 8])
Q_B Shape: torch.Size([2, 4, 8])
Q değerleri aynı mı? False


In [61]:
Q_A_heads = Q_A.reshape(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)

K_A_heads = K_A.reshape(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)

V_A_heads = V_A.reshape(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)


Q_B_heads = Q_B.reshape(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)

K_B_heads = K_B.reshape(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)

V_B_heads = V_B.reshape(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)


print("Q_A:", Q_A_heads.shape)
print("K_A:", K_A_heads.shape)
print("V_A:", V_A_heads.shape)

print("*"*30)

print("Q_B:", Q_B_heads.shape)
print("K_B:", K_B_heads.shape)
print("V_B:", V_B_heads.shape)


Q_A: torch.Size([2, 2, 4, 4])
K_A: torch.Size([2, 2, 4, 4])
V_A: torch.Size([2, 2, 4, 4])
******************************
Q_B: torch.Size([2, 2, 4, 4])
K_B: torch.Size([2, 2, 4, 4])
V_B: torch.Size([2, 2, 4, 4])


In [62]:
scores_A = torch.matmul(
    Q_A_heads,
    K_A_heads.transpose(-2, -1)
)

scaled_scores_A = scores_A / math.sqrt(head_dim)


scores_B = torch.matmul(
    Q_B_heads,
    K_B_heads.transpose(-2, -1)
)

scaled_scores_B = scores_B / math.sqrt(head_dim)


print("A Shape:", scaled_scores_A.shape)
print("B Shape:", scaled_scores_B.shape)

print("Scores aynı mı?", torch.allclose(scaled_scores_A, scaled_scores_B))

A Shape: torch.Size([2, 2, 4, 4])
B Shape: torch.Size([2, 2, 4, 4])
Scores aynı mı? False


In [64]:
masked_scores_A = scaled_scores_A.masked_fill(
    ~key_mask,
    float("-inf")
)


masked_scores_B = scaled_scores_B.masked_fill(
    ~key_mask,
    float("-inf")
)


print("A PAD Scores:")
print(masked_scores_A[1, :, :, 3])

print("B PAD Scores:")
print(masked_scores_B[1, :, :, 3])

A PAD Scores:
tensor([[-inf, -inf, -inf, -inf],
        [-inf, -inf, -inf, -inf]], grad_fn=<SelectBackward0>)
B PAD Scores:
tensor([[-inf, -inf, -inf, -inf],
        [-inf, -inf, -inf, -inf]], grad_fn=<SelectBackward0>)


In [66]:
weights_A = F.softmax(masked_scores_A, dim=-1)

weights_B = F.softmax(masked_scores_B, dim=-1)

print("A Shape:", weights_A.shape)
print("B Shape:", weights_B.shape)

print("\nDeney A — İlk Head, İlk Query:")
print(weights_A[1, 0, 0])

print("\nDeney B — İlk Head, İlk Query:")
print(weights_B[1, 0, 0])

print("\nAttention ağırlıkları aynı mı?")
print(torch.allclose(weights_A, weights_B))

A Shape: torch.Size([2, 2, 4, 4])
B Shape: torch.Size([2, 2, 4, 4])

Deney A — İlk Head, İlk Query:
tensor([0.4217, 0.2606, 0.3178, 0.0000], grad_fn=<SelectBackward0>)

Deney B — İlk Head, İlk Query:
tensor([0.3540, 0.2896, 0.3564, 0.0000], grad_fn=<SelectBackward0>)

Attention ağırlıkları aynı mı?
False


In [68]:
head_outputs_A = torch.matmul(weights_A, V_A_heads)

head_outputs_B = torch.matmul(weights_B, V_B_heads)

print("Output A Shape:", head_outputs_A.shape)
print("Output B Shape:", head_outputs_B.shape)

print("\nDeney A — İlk Token:")
print(head_outputs_A[0, 0, 0])

print("\nDeney B — İlk Token:")
print(head_outputs_B[0, 0, 0])

print("\nHead Outputs aynı mı?")
print(torch.allclose(head_outputs_A, head_outputs_B))

Output A Shape: torch.Size([2, 2, 4, 4])
Output B Shape: torch.Size([2, 2, 4, 4])

Deney A — İlk Token:
tensor([0.1205, 0.0750, 0.0501, 0.0320], grad_fn=<SelectBackward0>)

Deney B — İlk Token:
tensor([-0.0312, -0.5938, -0.2324, -0.0368], grad_fn=<SelectBackward0>)

Head Outputs aynı mı?
False


In [69]:
concat_A = head_outputs_A.transpose(1, 2).contiguous().reshape(
    batch_size, seq_len, d_model
)

concat_B = head_outputs_B.transpose(1, 2).contiguous().reshape(
    batch_size, seq_len, d_model
)

print("Concat A:", concat_A.shape)
print("Concat B:", concat_B.shape)

Concat A: torch.Size([2, 4, 8])
Concat B: torch.Size([2, 4, 8])


In [70]:
output_A = W_o(concat_A)

output_B = W_o(concat_B)

print("Output A Shape:", output_A.shape)
print("Output B Shape:", output_B.shape)

print("\nDeney A — İlk Token:")
print(output_A[0, 0])

print("\nDeney B — İlk Token:")
print(output_B[0, 0])

print("\nOutputs aynı mı?")
print(torch.allclose(output_A, output_B))

Output A Shape: torch.Size([2, 4, 8])
Output B Shape: torch.Size([2, 4, 8])

Deney A — İlk Token:
tensor([ 0.0928, -0.0312,  0.0803, -0.1119,  0.0724, -0.0335,  0.0509,  0.0183],
       grad_fn=<SelectBackward0>)

Deney B — İlk Token:
tensor([ 0.1567,  0.2253, -0.0919,  0.1568, -0.0110, -0.0073, -0.0113, -0.0148],
       grad_fn=<SelectBackward0>)

Outputs aynı mı?
False


In [71]:
print("Head 1 Attention Weights:")
print(weights_B[0, 0, 0])

print("\nHead 2 Attention Weights:")
print(weights_B[0, 1, 0])

Head 1 Attention Weights:
tensor([0.2825, 0.0860, 0.3881, 0.2435], grad_fn=<SelectBackward0>)

Head 2 Attention Weights:
tensor([0.3183, 0.2679, 0.1587, 0.2551], grad_fn=<SelectBackward0>)


In [72]:
print("Without Positional Encoding:")
print(output_A[0, 0])

print("\nWith Positional Encoding:")
print(output_B[0, 0])

print("\nOutputs aynı mı?")
print(torch.allclose(output_A[0, 0], output_B[0, 0]))

Without Positional Encoding:
tensor([ 0.0928, -0.0312,  0.0803, -0.1119,  0.0724, -0.0335,  0.0509,  0.0183],
       grad_fn=<SelectBackward0>)

With Positional Encoding:
tensor([ 0.1567,  0.2253, -0.0919,  0.1568, -0.0110, -0.0073, -0.0113, -0.0148],
       grad_fn=<SelectBackward0>)

Outputs aynı mı?
False


In [73]:
assert X.shape == (2, 4, 8)

assert Q_heads.shape == (2, 2, 4, 4)
assert K_heads.shape == (2, 2, 4, 4)
assert V_heads.shape == (2, 2, 4, 4)

assert scores.shape == (2, 2, 4, 4)

assert concatenated_output.shape == (2, 4, 8)
assert multihead_output.shape == (2, 4, 8)

assert concat_A.shape == (2, 4, 8)
assert concat_B.shape == (2, 4, 8)

assert output_A.shape == (2, 4, 8)
assert output_B.shape == (2, 4, 8)

print("Test 1 — Multi-Head Shapes: PASSED")


for weights in [attention_weights, weights_A, weights_B]:
    assert torch.allclose(
        weights[1, :, :, 3],
        torch.zeros_like(weights[1, :, :, 3]),
        atol=1e-6
    )

    assert torch.allclose(
        weights.sum(dim=-1),
        torch.ones_like(weights.sum(dim=-1)),
        atol=1e-6
    )

print("Test 2 — Padding Mask: PASSED")


assert PE.shape == (4, 8)

assert torch.isfinite(PE).all().item()

for i in range(PE.shape[0]):
    for j in range(i + 1, PE.shape[0]):
        assert not torch.allclose(PE[i], PE[j])

print("Test 3 — Positional Encoding: PASSED")

Test 1 — Multi-Head Shapes: PASSED
Test 2 — Padding Mask: PASSED
Test 3 — Positional Encoding: PASSED
